# 01 Fetch Census boundaries and ACS data

In [5]:
from pathlib import Path
import io, json, os, zipfile
from datetime import date

import pandas as pd
import requests
from dotenv import load_dotenv

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
KEY = os.getenv("CENSUS_API_KEY")
if not KEY:
    raise RuntimeError("CENSUS_API_KEY not found: copy .env.example to .env and add your key")

## County boundaries (cartographic, 1:500k)

In [6]:
BOUNDARY_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_500k.zip"
zip_path = RAW / "cb_2024_us_county_500k.zip"
shp_dir = RAW / "cb_2024_us_county_500k"

if not zip_path.exists():
    r = requests.get(BOUNDARY_URL, timeout=120)
    r.raise_for_status()
    zip_path.write_bytes(r.content)
    print("downloaded", zip_path.name)
else:
    print("cached", zip_path.name)

if not shp_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(shp_dir)
print(sorted(p.name for p in shp_dir.iterdir()))

cached cb_2024_us_county_500k.zip
['cb_2024_us_county_500k.cpg', 'cb_2024_us_county_500k.dbf', 'cb_2024_us_county_500k.prj', 'cb_2024_us_county_500k.shp', 'cb_2024_us_county_500k.shp.ea.iso.xml', 'cb_2024_us_county_500k.shp.iso.xml', 'cb_2024_us_county_500k.shx']


## ACS 5-year county tables

One call per vintage with `for=county:*` returns every county in the country.
Values arrive as strings; they are converted in `02_build`.

In [7]:
VARS = {
    2019: ["B25077_001E", "B19013_001E", "B01003_001E"],
    2024: ["B25077_001E", "B19013_001E", "B01003_001E", "B08006_001E", "B08006_017E"],
}

def fetch_acs(year, variables):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": "NAME," + ",".join(variables), "for": "county:*"}
    if KEY:
        params["key"] = KEY
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    rows = r.json()
    return pd.DataFrame(rows[1:], columns=rows[0])

log = {"access_date": str(date.today()), "rows": {}}
for year, variables in VARS.items():
    out = RAW / f"acs5_{year}_county.csv"
    if out.exists():
        df = pd.read_csv(out, dtype=str)
        print(f"cached   acs5 {year}: {len(df):,} counties")
    else:
        df = fetch_acs(year, variables)
        df.to_csv(out, index=False)
        print(f"fetched  acs5 {year}: {len(df):,} counties")
    log["rows"][year] = len(df)

(RAW / "fetch_log.json").write_text(json.dumps(log, indent=2))
df.head()

cached   acs5 2019: 3,220 counties
cached   acs5 2024: 3,222 counties


,NAME,B25077_001E,B19013_001E,B01003_001E,B08006_001E,B08006_017E,state,county
0,"Autauga County, Alabama",207200,72481,59947,27033,1799,01,001
1,"Baldwin County, Alabama",316900,78775,246989,112467,13035,01,003
2,"Barbour County, Alabama",112900,46042,24643,8469,553,01,005
3,"Bibb County, Alabama",145700,52541,22130,7944,379,01,007
4,"Blount County, Alabama",175200,64190,59518,25492,1564,01,009
